# 02 — Bayesian Uncertainty: Laplace + Fisher + σ(Q) Propagation

In notebook **01** you ran `autocalibrate_pv` and got a MAP geometry.
In this notebook you'll quantify the uncertainty on that geometry,
then propagate it into a per-Q-bin σ that downstream pipelines
(radial integration, PDF, Rietveld) can consume.

Three things you should know up front:

1. **The Laplace approximation** assumes the posterior near the MAP
   is Gaussian.  For prior-free fits this is a reasonable bet; for
   strong-prior fits you may want NUTS instead (see notebook 04 and
   the §"NUTS vs Laplace" subsection of the paper).

2. **σ on a parameter is meaningless without context** — gauge
   nullspaces (e.g. the (L_sd, p_x) multiplicative gauge if you
   refine pixel size with no prior) make the marginal σ go to
   infinity.  The framework reports honest σ; if a parameter is
   data-rank-deficient, you'll see it in the Fisher eigenvalues.

3. **σ on geometry isn't what your downstream pipeline wants** —
   it wants σ on the integrated quantity (Q, intensity, lattice
   constant, strain).  We propagate via the Jacobian chain rule.


In [1]:
import os, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import math
import numpy as np
import torch
from PIL import Image

from midas_calibrate.params import CalibrationParams as V1Params
from midas_calibrate_v2.compat.from_v1 import spec_from_v1_file
from midas_calibrate_v2.pipelines.single_pv import autocalibrate_pv
from midas_calibrate_v2.loss.pseudo_strain import pseudo_strain_residual
from midas_calibrate_v2.inference.laplace import fisher_at_map

BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
PARAMS = BASE / 'refined_MIDAS_params_Ceria_63keV_900mm_100x100_0p5s_aero_0.txt'
IMAGE  = BASE / 'Ceria_63keV_900mm_100x100_0p5s_aero_0_001137.tif'

v1 = V1Params.from_file(PARAMS)
if v1.RBinSize <= 0: v1.RBinSize = 0.25
if v1.EtaBinSize <= 0: v1.EtaBinSize = 5.0
v1.MaxRingRad = max(v1.MaxRingRad, v1.RhoD / max(v1.pxY, 1.0))
v1.Width = max(v1.Width, 800.0)
image = np.array(Image.open(str(IMAGE))).astype(np.float64)[::-1, :].copy()
spec = spec_from_v1_file(PARAMS)

t0 = time.time()
res = autocalibrate_pv(
    v1, image, spec=spec,
    n_iter=4, half_window_px=8.0, snr_min=8.0,
    trim_mode='stratified_multfactor', trim_residual_pct=5.0,
    reuse_fits=True, lm_max_iter=300, verbose=False,
    distribution_report=False,
)
print(f'pipeline: {time.time()-t0:.1f}s, '
      f'final strain {res.history[-1].mean_strain_uE:.2f} µε')


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

pipeline: 18.4s, final strain 18.48 µε


## Full Laplace covariance via `fisher_at_map`

`fisher_at_map` returns a `LaplaceResult` carrying:
- `map_refined` — the MAP parameter values (refined-only, packed)
- `cov` — the full N × N covariance matrix (in the spec's logit-bounded x-space)
- `sigma_per_dim` — sqrt(diag(cov)), per-parameter σ
- `refined_names`, `refined_offsets`, `refined_sizes` — index map

The empirical `sigma_r` (per-fit residual stddev at MAP) is the
noise scale used to normalise the Fisher.


In [2]:
fits = res.fits_final
def residual_fn(unp):
    return pseudo_strain_residual(
        fits.Y_pix, fits.Z_pix, fits.ring_two_theta_deg, unp,
        rho_d=fits.rho_d, weights=fits.weights,
        ring_idx=fits.ring_idx,
        ring_d_spacing_A=fits.ring_d_spacing_A,
    )
with torch.no_grad():
    r = residual_fn(res.unpacked)
    sigma_r = float(((r * r).mean()) ** 0.5)

lap = fisher_at_map(spec, residual_fn, res.unpacked,
                    sigma_r=sigma_r, ridge=1e-9,
                    dtype=torch.float64, device='cpu')

cov = lap.cov.detach().cpu().numpy()
print(f'Laplace cov shape: {cov.shape}, condition number: '
      f'{np.linalg.cond(cov):.2e}')
print(f'Refined parameters: {lap.refined_names}')


Laplace cov shape: (20, 20), condition number: 2.16e+16
Refined parameters: ['Lsd', 'BC_y', 'BC_z', 'ty', 'tz', 'a2', 'a4', 'iso_R2', 'phi4', 'iso_R6', 'iso_R4', 'phi2', 'a1', 'phi1', 'a3', 'phi3', 'a5', 'phi5', 'a6', 'phi6']


## Reading per-parameter σ in physical units

The Fisher is built in a logit-reparameterised u-space (so the LM
trust-region updates don't have to respect parameter bounds).  The
σ values are transformed back to physical x-space via the
sigmoid Jacobian.

Below: σ on the headline geometry parameters, with units.


In [3]:
def _flat(lap):
    out = []
    for n, o, s in zip(lap.refined_names, lap.refined_offsets, lap.refined_sizes):
        for k in range(s):
            out.append(f'{n}[{k}]' if s > 1 else n)
    return out

flat = _flat(lap)
sigma_arr = lap.sigma_per_dim.detach().cpu().numpy()
map_arr = lap.map_refined.detach().cpu().numpy()

UNITS = {'Lsd': 'µm', 'BC_y': 'px', 'BC_z': 'px', 'ty': '°', 'tz': '°',
         'pxY': 'µm', 'pxZ': 'µm', 'Wavelength': 'Å', 'Parallax': 'µm'}
print(f'{"param":<12s}  {"MAP":>16s}  {"σ":>14s}  unit')
for nm, mp, sg in zip(flat, map_arr, sigma_arr):
    base = nm.split('[')[0]
    unit = UNITS.get(base, '')
    print(f'  {nm:<10s}  {mp:>+16.6e}  {sg:>14.3e}  {unit}')


param                      MAP               σ  unit
  Lsd            +8.958988e+05       1.607e+00  µm
  BC_y           +1.446973e+03       7.289e-04  px
  BC_z           +1.468911e+03       7.314e-04  px
  ty             -3.073929e-01       6.993e-04  °
  tz             +3.899691e-01       6.980e-04  °
  a2             +3.431613e-04       1.778e-06  
  a4             +1.676060e-04       5.022e-06  
  iso_R2         -9.323869e-05       3.420e-05  
  phi4           -7.253492e+00       1.720e+00  
  iso_R6         -9.151903e-04       2.525e-04  
  iso_R4         +1.209033e-03       1.742e-04  
  phi2           -4.278301e+00       2.962e-01  
  a1             +4.296078e-04       1.652e-05  
  phi1           +1.101152e+02       2.204e+00  
  a3             -1.263484e-04       3.024e-06  
  phi3           +1.275762e+02       1.372e+00  
  a5             -9.635199e-05       8.205e-06  
  phi5           +2.771147e+02       4.880e+00  
  a6             +4.401770e-06       1.323e-05  
  phi6  

## σ(Q) propagation for downstream pipelines

`Q = (4π/λ) sin(θ)` is what radial integration, PDF analysis
(`PDFgetX3`, `diffpy`), and Rietveld refinement (`GSAS-II`,
`TOPAS`) consume.  Per-Q-bin uncertainty needs the chain rule:
$$\sigma^2(Q_k) = J_Q^\top \, \mathrm{Cov} \, J_Q$$
where $J_Q$ is the Jacobian of $Q_k$ w.r.t. the refined geometry
parameters, evaluated at the MAP.

For the headline Varex configuration (only L_sd refined among
parameters affecting Q_k), the chain reduces to
$\sigma(Q)/Q \approx \sigma(L_{\mathrm{sd}})/L_{\mathrm{sd}}$ —
constant in ppm across all rings.


In [4]:
rt = fits.rt
two_theta_deg = np.array(rt.two_theta_deg)
lam_A = float(res.unpacked['Wavelength'])
Lsd_um = float(res.unpacked['Lsd'])
pxY_um = float(res.unpacked['pxY'])
sigma_Lsd_um = float(lap.sigma_per_dim[flat.index('Lsd')])

print(f'{"ring":>4s}  {"2θ (°)":>7s}  {"Q (Å⁻¹)":>9s}  '
      f'{"d (Å)":>7s}  {"σ(Q) (Å⁻¹)":>14s}  {"σ(Q)/Q (ppm)":>14s}')
for k, tt_deg in enumerate(two_theta_deg):
    if tt_deg <= 0:
        continue
    tt_rad = math.radians(tt_deg)
    th_rad = 0.5 * tt_rad
    R_obs_px = Lsd_um * math.tan(tt_rad) / pxY_um
    Q_k = (4.0 * math.pi / lam_A) * math.sin(th_rad)
    d_k = 2.0 * math.pi / Q_k
    # ∂Q/∂L_sd at fixed R_obs (small-angle dominant term)
    dtt_dLsd = -R_obs_px * pxY_um / (Lsd_um**2 + (R_obs_px * pxY_um)**2)
    dth_dLsd = 0.5 * dtt_dLsd
    dQ_dLsd  = (4.0 * math.pi / lam_A) * math.cos(th_rad) * dth_dLsd
    sigma_Q = abs(dQ_dLsd) * sigma_Lsd_um
    sigma_Q_ppm = sigma_Q / Q_k * 1e6
    print(f'  {k:>2d}  {tt_deg:>7.2f}  {Q_k:>9.3f}  {d_k:>7.3f}  '
          f'{sigma_Q:>14.3e}  {sigma_Q_ppm:>14.2f}')


ring   2θ (°)    Q (Å⁻¹)    d (Å)      σ(Q) (Å⁻¹)    σ(Q)/Q (ppm)
   0     3.61      2.011    3.124       3.595e-06            1.79
   1     4.17      2.322    2.706       4.148e-06            1.79
   2     5.90      3.284    1.913       5.842e-06            1.78
   3     6.91      3.851    1.632       6.830e-06            1.77
   4     7.22      4.022    1.562       7.127e-06            1.77
   5     8.34      4.644    1.353       8.196e-06            1.76
   6     9.09      5.061    1.242       8.905e-06            1.76
   7     9.33      5.192    1.210       9.127e-06            1.76
   8    10.22      5.688    1.105       9.958e-06            1.75
   9    10.84      6.033    1.041       1.053e-05            1.75
  10    10.84      6.033    1.041       1.053e-05            1.75
  11    11.81      6.568    0.957       1.141e-05            1.74
  12    12.35      6.869    0.915       1.189e-05            1.73
  13    12.53      6.966    0.902       1.205e-05            1.73
  14    12

The `σ(Q)/Q` is essentially constant across rings (Q ∝ 1/L_sd at
small angle, so the ratio is just `σ(L_sd)/L_sd`).  For the Varex
headline, this is ~0.78 ppm — far below typical PDF/Rietveld bin
widths (`ΔQ ~ 10⁻³ Å⁻¹` is 10⁵ ppm at Q=1 Å⁻¹).

If you also refine pxY/pxZ or Wavelength, the chain gets more
columns and the σ(Q) inflates accordingly.  See
`dev/paper/runners/run_sigma_q_propagation.py` for the full
analysis with all the Jacobian terms.

## When to use NUTS instead

The Laplace approximation breaks down when:
- A strong **prior is wired into the LM closure** (the σ-pull
  equilibrium MAP has a steep local Hessian that under-counts the
  actual posterior spread — see `tab:nuts_vs_laplace` in the
  paper, where Laplace under-counts by 1.5–17× under an L_sd prior).
- The posterior is **multi-modal** (e.g., a basin escape between
  two local minima of the LM landscape).
- You're refining **phase parameters** of harmonic distortion,
  which can have a sign ambiguity that Gaussian-around-MAP misses.

For these regimes, sample directly with HMC.  See
`dev/paper/runners/run_nuts_vs_laplace.py` for the canonical
example using `inference.hmc.hmc_run` (pyro backend; `pip install
pyro-ppl` if not present).
